In [1]:
import os
import glob
import math
import folium
from PIL import Image
from PIL.ExifTags import TAGS, GPSTAGS

# ==============================================================================
# 1. SPÉCIFICATIONS OPTIQUES DJI FC3411 (M3E)
# ==============================================================================
SENSOR_WIDTH_MM = 17.3
SENSOR_HEIGHT_MM = 13.0
FOCAL_LENGTH_MM = 12.29

# ==============================================================================
# 2. EXTRACTION DES DONNÉES GPS EXIF
# ==============================================================================
def get_exif_data(image_path):
    """Extrait la latitude, longitude et altitude relatives depuis l'EXIF."""
    img = Image.open(image_path)
    exif = img._getexif()
    if not exif:
        return None
    
    gps_info = {}
    for tag, value in exif.items():
        decoded = TAGS.get(tag, tag)
        if decoded == "GPSInfo":
            for g_tag in value:
                g_decoded = GPSTAGS.get(g_tag, g_tag)
                gps_info[g_decoded] = value[g_tag]
                
    if not gps_info:
        return None

    def convert_to_degrees(value):
        d, m, s = value
        return d + (m / 60.0) + (s / 3600.0)

    lat = convert_to_degrees(gps_info['GPSLatitude'])
    if gps_info['GPSLatitudeRef'] != 'N':
        lat = -lat
        
    lon = convert_to_degrees(gps_info['GPSLongitude'])
    if gps_info['GPSLongitudeRef'] != 'E':
        lon = -lon
        
    alt = gps_info.get('GPSAltitude', 25.0) # Altitude par défaut si absente
    return lat, lon, alt

# ==============================================================================
# 3. CALCUL DES BOUNDS (COINS DE L'IMAGE AU SOL)
# ==============================================================================
def calculate_image_bounds(lat, lon, alt_m, img_width_px, img_height_px):
    """Calcule les coordonnées Sud-Ouest et Nord-Est de l'emprise au sol."""
    # GSD en mètres
    gsd_w = (alt_m * SENSOR_WIDTH_MM) / (FOCAL_LENGTH_MM * img_width_px)
    gsd_h = (alt_m * SENSOR_HEIGHT_MM) / (FOCAL_LENGTH_MM * img_height_px)
    
    half_width_m = (img_width_px * gsd_w) / 2.0
    half_height_m = (img_height_px * gsd_h) / 2.0
    
    # Conversion mètres -> Delta GPS
    lat_deg_val = half_height_m / 111111.0
    lon_deg_val = half_width_m / (111111.0 * math.cos(math.radians(lat)))
    
    south_west = [lat - lat_deg_val, lon - lon_deg_val]
    north_east = [lat + lat_deg_val, lon + lon_deg_val]
    
    return [south_west, north_east]

# ==============================================================================
# 4. CRÉATION DE LA MOSAÏQUE CARTOGRAPHIQUE
# ==============================================================================
def create_raw_mosaic(images_folder, output_html="mosaique_brute.html", crop_center=True):
    """
    Lit toutes les images d'un dossier et crée une carte interactive Leaflet.
    Si crop_center=True, on ne garde que le centre de la photo pour éliminer les chevauchements.
    """
    image_paths = glob.glob(os.path.join(images_folder, "*.jpg")) + glob.glob(os.path.join(images_folder, "*.JPG"))
    
    if not image_paths:
        print("❌ Aucune image trouvée dans le dossier.")
        return

    first_lat, first_lon, _ = get_exif_data(image_paths[0])
    m = folium.Map(location=[first_lat, first_lon], zoom_start=19, max_zoom=23)

    os.makedirs("cropped_tiles", exist_ok=True)

    print(f"⚙️ Traitement de {len(image_paths)} images brutes...")

    for img_path in image_paths:
        coords = get_exif_data(img_path)
        if not coords:
            continue
            
        lat, lon, alt = coords
        filename = os.path.basename(img_path)
        
        with Image.open(img_path) as img:
            w, h = img.size
            
            if crop_center:
                # On rogne 30% des bords pour ne garder que 40% du centre (Nadir pur)
                # Cela supprime naturellement le chevauchement excessif de 80%
                crop_margin_w = int(w * 0.30)
                crop_margin_h = int(h * 0.30)
                
                cropped_img = img.crop((
                    crop_margin_w, crop_margin_h, 
                    w - crop_margin_w, h - crop_margin_h
                ))
                
                tile_path = os.path.join("cropped_tiles", f"tile_{filename}")
                cropped_img.save(tile_path, quality=95)
                
                # Ajustement de l'emprise au sol pour l'image rognée
                bounds = calculate_image_bounds(lat, lon, alt, w * 0.4, h * 0.4)
            else:
                tile_path = img_path
                bounds = calculate_image_bounds(lat, lon, alt, w, h)

            # Overlay direct de l'image sur la carte (Sans aucun blending ni interpolation !)
            folium.raster_layers.ImageOverlay(
                image=tile_path,
                bounds=bounds,
                opacity=1.0,
                interactive=True,
                cross_origin=False
            ).add_to(m)

    m.save(output_html)
    print(f"✅ Mosaïque générée avec succès : Ouvrez '{output_html}' dans votre navigateur.")

# ==============================================================================
# EXECUTION
# ==============================================================================
if __name__ == "__main__":
    # Remplacez par le chemin vers votre dossier d'images drone
    DOSSIER_IMAGES = "E:\\PixelOdyssey\\2. Raw data\\3. Santa Luzia\\Raw pictures\\Santa Luzia W4 - mar - transect (22 - 25) avant ramassage (done)"
    
    create_raw_mosaic(DOSSIER_IMAGES, output_html="mosaique_drone.html", crop_center=True)

⚙️ Traitement de 90 images brutes...
✅ Mosaïque générée avec succès : Ouvrez 'mosaique_drone.html' dans votre navigateur.
